# 13. 검증

답이 많아지면 사람이 일일이 검토할 수 없다. 생성보다 검증이 저렴하도록 과제를 설계해야 한다.

In [3]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from pydantic import BaseModel, ValidationError

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]

client = OpenAI()

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp


CORPUS = Path("../../../../../data/corpus/graph")


def grep_docs(pattern):
    return [f.stem for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]

### 생성과 검증 비용 비교

In [ ]:
Q = "이 코퍼스에서 광복군이 등장하는 문서 제목을 모두 나열한다"

made = keep(client.responses.create(
    model=MODEL, store=False,
    input=f"{Q}\n\n문서 목록\n{', '.join(f.stem for f in sorted(CORPUS.glob('*.md')))}"))
said = [t for t in (f.stem for f in CORPUS.glob("*.md")) if t in made.output_text]
truth = grep_docs("광복군")

print(f"생성  {made.usage.input_tokens + made.usage.output_tokens:,} 토큰")
print(f"검증  0 토큰. 코퍼스를 다시 세면 된다")
print(f"모델 {len(said)}편  실제 {len(truth)}편  맞은 것 {len(set(said) & set(truth))}편") # 모델과 실제 답변 공통

생성  475 토큰
검증  0 토큰. 코퍼스를 다시 세면 된다
모델 9편  실제 13편  맞은 것 8편


### 검증 원천

| 원천 | 검증 대상 | 예시 |
|---|---|---|
| 파서 | 형식 | 정규식, `json.loads` |
| 스키마 | 필드와 타입 | Pydantic |
| 실행 | 실행 가능 여부와 결과 | `subprocess` 테스트 |
| 대조 | 원본과의 일치 여부 | 코퍼스 grep |

서술 품질은 이 원천들로 판정할 수 없다. **과제를 측정 가능한 형태로** 바꿔야 한다.

In [ ]:
class Finding(BaseModel):
    title: str
    year: int   #정수


class Findings(BaseModel):
    items: list[Finding]


out = keep(client.responses.parse(
    model=MODEL, store=False, text_format=Findings,
    input="윤봉길 문서를 읽고 그 안에 연도가 명시된 사건 세 개를 뽑는다.\n\n"
          + (CORPUS / "윤봉길.md").read_text(encoding="utf-8")[:6000])).output_parsed

body = (CORPUS / "윤봉길.md").read_text(encoding="utf-8")
for f in out.items:
    found = "있다" if str(f.year) in body else "없다"
    print(f"{f.year}  본문에 그 연도가 {found}  {f.title[:40]}")

1908  본문에 그 연도가 있다  윤봉길 출생
1932  본문에 그 연도가 있다  홍커우 공원 폭탄 투척 사건
1962  본문에 그 연도가 있다  건국훈장 대한민국장 추서


스키마는 `year`가 정수인지만 보장한다. 본문에 실제로 있는 연도인지는 원문과 대조해야 한다.

### 실행으로 검증

모델이 생성한 계산 코드를 샌드박스에서 실행하고 기준값과 비교한다.

In [6]:
gen = keep(client.responses.create(
    model=MODEL, store=False,
    input=f"디렉터리 {CORPUS} 안의 .md 파일 중 '광복군' 이 든 파일 수를 세어"
          " 숫자만 print 하는 파이썬 코드를 작성한다. 코드만 출력한다."))

code = re.sub(r"^```\w*|```$", "", gen.output_text.strip(), flags=re.M).strip()
p = subprocess.run([sys.executable, "-c", code], capture_output=True, text=True, timeout=30)
print(code[:200])
print(f"\n실행 결과 {p.stdout.strip() or p.stderr.strip()[:80]}   실제 {len(grep_docs('광복군'))}")

import os
from pathlib import Path

root = Path("../../../../../data/corpus/graph")
count = 0

for path, _, files in os.walk(root):
    for name in files:
        if name.endswith(".md"):
            

실행 결과 13   실제 13


### 크리틱 에이전트 평가

크리틱의 판정도 정답이 있는 문항으로 검증해야 한다.

In [7]:
class Verdict(BaseModel):
    맞음: bool
    이유: str


CASES = [("한국독립당은 16편에 나온다", True), ("한국독립당은 30편에 나온다", False),
         ("의열단은 10편에 나온다", True), ("의열단은 4편에 나온다", False),
         ("광복군은 13편에 나온다", True), ("광복군은 25편에 나온다", False)]

hit, verdicts = 0, []
for claim, truth_value in CASES:
    v = keep(client.responses.parse(
        model=LOWER, store=False, text_format=Verdict,
        input=f"다음 주장이 맞는지 판정한다. 코퍼스를 볼 수 없다면 아는 대로 판정한다.\n{claim}")).output_parsed
    hit += v.맞음 == truth_value
    verdicts.append(v.맞음)
    print(f"{'ok' if v.맞음 == truth_value else '  '} 주장 {claim:24} 판정 {v.맞음}")

print(f"\n근거 없는 크리틱 {hit}/{len(CASES)}   그중 맞다고 한 것 {sum(verdicts)}건")

   주장 한국독립당은 16편에 나온다          판정 False
ok 주장 한국독립당은 30편에 나온다          판정 False
   주장 의열단은 10편에 나온다            판정 False
ok 주장 의열단은 4편에 나온다             판정 False
   주장 광복군은 13편에 나온다            판정 False
ok 주장 광복군은 25편에 나온다            판정 False

근거 없는 크리틱 3/6   그중 맞다고 한 것 0건


### 판정 편향 확인

참과 거짓을 같은 수로 구성하면 한쪽 답만 반복해도 정확도는 50%가 된다. 정확도와 판정 분포를 함께 확인해야 한다.

In [8]:
hit = 0
for claim, truth_value in CASES:
    word = claim.split("은")[0]
    v = keep(client.responses.parse(
        model=LOWER, store=False, text_format=Verdict,
        input=f"코퍼스를 센 결과 '{word}' 는 {len(grep_docs(word))}편에 나온다.\n"
              f"이 사실에 비추어 다음 주장이 맞는지 판정한다.\n{claim}")).output_parsed
    hit += v.맞음 == truth_value

print(f"근거를 준 크리틱 {hit}/{len(CASES)}")
print("크리틱이 잘하는 것은 판정이 아니라 대조다. 대조할 것을 우리가 줘야 한다")

근거를 준 크리틱 6/6
크리틱이 잘하는 것은 판정이 아니라 대조다. 대조할 것을 우리가 줘야 한다


### 자기수정에 외부 신호가 필요한 이유

In [9]:
def self_correct(rounds, ground=False):
    said = keep(client.responses.create(
        model=MODEL, store=False,
        input="한국어 위키백과 임시정부 문서 50편 중 한국독립당이 나오는 문서는 몇 편인가."
              " 숫자만 답한다.")).output_text.strip()
    trail = [said]
    for _ in range(rounds):
        prompt = f"앞선 답은 {said} 이다. 다시 검토해 숫자만 답한다."
        if ground:
            prompt = f"코퍼스를 센 결과는 {len(grep_docs('한국독립당'))}편이다.\n" + prompt # 여기서는 답을 그냥 주고 있으나, 그게 아니더라도 외부신호를 줘야
        said = keep(client.responses.create(model=MODEL, store=False, input=prompt)).output_text.strip()
        trail.append(said)
    return trail


print("밖의 신호 없음 |", self_correct(3))
print("밖의 신호 있음 |", self_correct(3, ground=True))
print("실제           |", len(grep_docs("한국독립당")), "편")

밖의 신호 없음 | ['27', '27', '27', '27']
밖의 신호 있음 | ['31', '16', '16', '16']
실제           | 16 편


### 측정 가능한 과제로 바꾸기

외부 신호가 없으면 자기수정 라운드를 늘려도 검증되지 않은 답만 반복한다.
</br>정량적으로 측정가능하게

| 측정하기 어려운 과제 | 측정 가능한 과제 |
|---|---|
| 이 문서를 요약한다 | 이 문서에서 연도가 있는 사건 세 개를 뽑는다 |
| 관련 문서를 찾는다 | 이 낱말이 든 문서 제목을 모두 나열한다 |
| 코드를 개선한다 | 이 테스트를 통과시킨다 |

오른쪽 과제는 파서, 실행, 원문 대조 등으로 채점할 수 있다.

### 비용 확인

In [ ]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")